# Chatbot RNN-LSTM — Répliques de films
**Stack :** ConvoKit · NLTK · TensorFlow/Keras · ipywidgets  
**Modèle :** LSTM simple (classification d'intention → réponse)

---
## Étape 0 — Installation des dépendances
Lance cette cellule **une seule fois**, puis redémarre le kernel.

In [ ]:
import sys
!{sys.executable} -m pip install convokit nltk tensorflow ipywidgets --quiet
print('✅ Installation terminée — redémarre le kernel si première exécution')

## Étape 1 — Imports

In [ ]:
import os, json, random, pickle
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from convokit import Corpus, download

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, LSTM, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import ipywidgets as widgets
from IPython.display import display, clear_output

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab', quiet=True)

lemmatizer = WordNetLemmatizer()
print('✅ Imports OK — TF version:', tf.__version__)

## Étape 2 — Chargement du dataset Movie-Corpus
Le corpus sera téléchargé automatiquement dans le dossier du projet.

In [ ]:
# Dossier projet
PROJECT_DIR = r'C:\Users\prosa\Desktop\IPSA PARIS\B3\projet_chatbot'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

print('📂 Dossier projet :', os.getcwd())
print('⏳ Téléchargement du corpus (peut prendre quelques minutes)...')

corpus = Corpus(filename=download('movie-corpus'))
print('✅ Corpus chargé')
print(f'   Conversations : {len(list(corpus.iter_conversations()))}')
print(f'   Utterances    : {len(list(corpus.iter_utterances()))}')

## Étape 3 — Extraction des paires (input → réponse)

In [ ]:
MAX_PAIRS   = 15000   # nb de paires à garder (augmente si ta machine le permet)
MIN_LEN     = 2       # longueur minimale en mots
MAX_LEN_TOK = 15      # longueur max en tokens pour le padding

def clean(text):
    """Minuscule + lemmatisation simple."""
    tokens = nltk.word_tokenize(text.lower())
    return ' '.join(lemmatizer.lemmatize(t) for t in tokens if t.isalpha())

pairs = []  # (input_clean, response_raw)

for convo in corpus.iter_conversations():
    utts = list(convo.iter_utterances())
    for i in range(len(utts) - 1):
        inp  = utts[i].text
        resp = utts[i + 1].text
        if inp and resp:
            ci = clean(inp)
            if len(ci.split()) >= MIN_LEN:
                pairs.append((ci, resp.strip()))
    if len(pairs) >= MAX_PAIRS:
        break

random.shuffle(pairs)
pairs = pairs[:MAX_PAIRS]

print(f'✅ {len(pairs)} paires extraites')
print('\nExemples :')
for inp, resp in pairs[:3]:
    print(f'  Q: {inp[:60]}')
    print(f'  R: {resp[:60]}')
    print()

## Étape 4 — Préparation des données pour le LSTM

Stratégie : on traite le problème comme de la **classification** — chaque réponse unique est une classe. Le LSTM prédit quelle réponse renvoyer.

In [ ]:
# 4a — Déduplique les réponses → classes
unique_responses = list(dict.fromkeys(r for _, r in pairs))  # ordre préservé
resp2idx = {r: i for i, r in enumerate(unique_responses)}
NUM_CLASSES = len(unique_responses)
print(f'Classes (réponses uniques) : {NUM_CLASSES}')

# 4b — Tokenisation des inputs
tokenizer = Tokenizer(oov_token='<OOV>')
inputs_raw = [inp for inp, _ in pairs]
tokenizer.fit_on_texts(inputs_raw)
VOCAB_SIZE = len(tokenizer.word_index) + 1
print(f'Vocabulaire : {VOCAB_SIZE} tokens')

sequences = tokenizer.texts_to_sequences(inputs_raw)
X = pad_sequences(sequences, maxlen=MAX_LEN_TOK, padding='post', truncating='post')

# 4c — Labels one-hot
y_idx = np.array([resp2idx[r] for _, r in pairs])
y = tf.keras.utils.to_categorical(y_idx, num_classes=NUM_CLASSES)

# 4d — Split train/val
split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f'Train : {X_train.shape} | Val : {X_val.shape}')

## Étape 5 — Construction du modèle LSTM

In [ ]:
EMBED_DIM = 64
LSTM_UNITS = 128

model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN_TOK),
    LSTM(LSTM_UNITS, return_sequences=False),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Étape 6 — Entraînement

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

print('\n✅ Entraînement terminé')

## Étape 7 — Sauvegarde du modèle et des artefacts

In [ ]:
model.save('chatbot_model.keras')

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open('responses.json', 'w', encoding='utf-8') as f:
    json.dump(unique_responses, f, ensure_ascii=False, indent=2)

meta = {
    'MAX_LEN_TOK': MAX_LEN_TOK,
    'NUM_CLASSES': NUM_CLASSES,
    'VOCAB_SIZE': VOCAB_SIZE
}
with open('meta.json', 'w') as f:
    json.dump(meta, f)

print('✅ Fichiers sauvegardés :')
for f in ['chatbot_model.keras', 'tokenizer.pkl', 'responses.json', 'meta.json']:
    print(f'   {f}')

## Étape 8 — Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100)
plt.show()
print('✅ Courbes sauvegardées → training_curves.png')

## Étape 9 — Chargement du modèle sauvegardé
*(Exécute cette cellule si tu reprends le notebook sans ré-entraîner)*

In [ ]:
import os, json, pickle
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('punkt',    quiet=True)
nltk.download('wordnet',  quiet=True)
nltk.download('punkt_tab',quiet=True)
lemmatizer = WordNetLemmatizer()

os.chdir(r'C:\Users\prosa\Desktop\IPSA PARIS\B3\projet_chatbot')

model_chat    = load_model('chatbot_model.keras')
with open('tokenizer.pkl', 'rb') as f:
    tok = pickle.load(f)
with open('responses.json', encoding='utf-8') as f:
    responses = json.load(f)
with open('meta.json') as f:
    meta = json.load(f)

MAX_LEN = meta['MAX_LEN_TOK']
print('✅ Modèle et artefacts chargés')

## Étape 10 — Fonction de prédiction

In [ ]:
def preprocess(text):
    tokens = nltk.word_tokenize(text.lower())
    return ' '.join(lemmatizer.lemmatize(t) for t in tokens if t.isalpha())

def predict_response(user_input, threshold=0.15, top_k=3):
    """
    Renvoie la réponse prédite par le LSTM.
    Si la confiance est trop basse, renvoie une réponse aléatoire parmi les top_k.
    """
    cleaned   = preprocess(user_input)
    seq       = tok.texts_to_sequences([cleaned])
    padded    = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    probs     = model_chat.predict(padded, verbose=0)[0]
    best_idx  = int(np.argmax(probs))
    confidence = float(probs[best_idx])

    if confidence < threshold:
        # réponse incertaine → pioche parmi les top_k
        top_indices = np.argsort(probs)[-top_k:]
        chosen = int(random.choice(top_indices))
        return responses[chosen], confidence

    return responses[best_idx], confidence

# Test rapide
r, c = predict_response('hello how are you')
print(f'Test → "{r}" (confiance : {c:.2%})')

## Étape 11 — Interface Chatbot (ipywidgets)
Écris un message dans le champ et appuie sur **Envoyer** ou **Entrée**.

In [ ]:
# ── UI ──────────────────────────────────────────────────────────────────────
chat_log = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd',
        min_height='320px',
        max_height='320px',
        overflow_y='auto',
        padding='10px',
        background_color='#fafafa'
    )
)

text_input = widgets.Text(
    placeholder='Écris un message…',
    layout=widgets.Layout(width='75%')
)

send_btn = widgets.Button(
    description='Envoyer',
    button_style='primary',
    layout=widgets.Layout(width='23%')
)

clear_btn = widgets.Button(
    description='Effacer',
    button_style='warning',
    layout=widgets.Layout(width='100px')
)

def html_msg(role, text, conf=None):
    if role == 'user':
        color = '#0078d4'
        align = 'right'
        label = '🧑 Toi'
        extra = ''
    else:
        color = '#107c10'
        align = 'left'
        label = '🤖 Bot'
        extra = f' <span style="font-size:0.75em;color:#999">(conf. {conf:.1%})</span>' if conf else ''
    return (
        f'<div style="text-align:{align};margin:6px 0">'
        f'<span style="font-weight:bold;color:{color}">{label}</span>{extra}<br>'
        f'<span style="background:white;border:1px solid #eee;border-radius:8px;'
        f'padding:5px 10px;display:inline-block;max-width:80%;text-align:left">'
        f'{text}</span></div>'
    )

def on_send(_=None):
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    text_input.value = ''
    response, conf = predict_response(user_msg)
    with chat_log:
        display(widgets.HTML(html_msg('user', user_msg)))
        display(widgets.HTML(html_msg('bot', response, conf)))

def on_clear(_):
    chat_log.clear_output()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_input.on_submit(on_send)  # Entrée = Envoyer

title = widgets.HTML('<h3 style="margin:0 0 8px">🎬 Movie Chatbot — LSTM</h3>')
input_row = widgets.HBox([text_input, send_btn])
footer = widgets.HBox([clear_btn])

display(widgets.VBox([title, chat_log, input_row, footer]))